In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'device used: PC GPU - {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"device used: MAC GPU - MPS")
else:
    device = torch.device("cpu")
    print(f"CPU used")

device used: MAC GPU - MPS


In [2]:
from transformers import AutoTokenizer
import pandas as pd

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def align_labels_tokens(labels, word_ids):
    new_labels = []
    current_word = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)

        elif word_id != current_word:
            current_word = word_id
            new_labels.append(current_word)

        else:
            new_labels.append(-100)

    return new_labels

In [4]:
sample_tokens = ['Experience','with', 'micro-services','like','Kubernetes']
sample_tags = ['O','O','B-KNOWLEDGE','O','B-KNOWLEDGE']

tokenize_input = tokenizer(sample_tokens, is_split_into_words=True)
word_ids = tokenize_input.word_ids()

print(tokenize_input)
print(word_ids)

aligned_labels = align_labels_tokens(sample_tags, word_ids)

subwords = tokenizer.convert_ids_to_tokens(tokenize_input["input_ids"])

print(f"{'SUBWORD':<15} | {'ORIGINAL WORD ID':<18} | {'ALIGNED LABEL'}")
print("------------------------------------------------------------------")
for subword, word_id, label in zip(subwords, word_ids, aligned_labels):
    print(f"{'subword':<15} | {str(word_id):<18} | {label}")

{'input_ids': [101, 15843, 1114, 17599, 118, 1826, 1176, 23209, 3169, 6097, 1279, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[None, 0, 1, 2, 2, 2, 3, 4, 4, 4, 4, None]
SUBWORD         | ORIGINAL WORD ID   | ALIGNED LABEL
------------------------------------------------------------------
subword         | None               | -100
subword         | 0                  | 0
subword         | 1                  | 1
subword         | 2                  | 2
subword         | 2                  | -100
subword         | 2                  | -100
subword         | 3                  | 3
subword         | 4                  | 4
subword         | 4                  | -100
subword         | 4                  | -100
subword         | 4                  | -100
subword         | None               | -100


In [5]:
df = pd.read_parquet("../data/df_cleaned.parquet")
df.head()

,idx,tokens,tags_skill,tags_knowledge,source,split,num_skills,num_knowledge,has_overlap,tokens_length
0,1,"[Senior, QA, Engineer, (, m/f/d, ), <ORGANIZAT...","[O, O, O, O, O, O, O]","[O, O, O, O, O, O, O]",tech,train,0,0,False,7
1,1,"[<ADDRESS>, <ADDRESS>, <ADDRESS>, <ADDRESS>, <...","[O, O, O, O, O]","[O, O, O, O, O]",tech,train,0,0,False,5
2,1,"[Date, posted:, 2021-07-14]","[O, O, O]","[O, O, O]",tech,train,0,0,False,3
3,1,"[Likes:, 0, Dislikes:, 0, Love:, 0]","[O, O, O, O, O, O]","[O, O, O, O, O, O]",tech,train,0,0,False,6
4,1,"[Job, description:]","[O, O]","[O, O]",tech,train,0,0,False,2


In [6]:
unique_skills = set(tag for row in df['tags_skill'] for tag in row)
unique_knowledge = set(tag for row in df['tags_knowledge'] for tag in row)

skills2id = {tag: i for i, tag in enumerate(unique_skills)}
knowledge2id = {tag: i for i, tag in enumerate(unique_knowledge)}

id2skills = {i: tag for tag,i in skills2id.items()}
id2knowledge = {i: tag for tag,i in knowledge2id.items()}

print(skills2id)
print(knowledge2id)


{'B': 0, 'O': 1, 'I': 2}
{'B': 0, 'O': 1, 'I': 2}


In [7]:
def tokenize_align(sample, label_col, label2id):
    tokenize_inputs = tokenizer(
        sample["tokens"],
        truncation = True,
        max_length = 512,
        is_split_into_words = True,
        return_overflowing_tokens = True,
        stride = 50
    )

    labels = []

    for i, seq_idx in enumerate(tokenize_inputs.pop("overflow_to_sample_mapping")):     # map overflowing chunks back
        word_ids = tokenize_inputs.word_ids(batch_index = i)
        org_labels = sample[label_col][seq_idx]

        label_ids = [label2id[label] for label in org_labels]       # convert string labels to IDs

        aligned = align_labels_tokens(label_ids,word_ids)
        labels.append(aligned)

    tokenize_inputs["labels"] = labels
    return tokenize_inputs

In [8]:
from datasets import Dataset
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

In [9]:
X_train = Dataset.from_pandas(df[df['split'] == "train"])
X_val = Dataset.from_pandas(df[df['split'] == "validation"])
X_test = Dataset.from_pandas(df[df['split'] == "test"])

columns_to_remove = X_train.column_names

train_tokenized_skill = X_train.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

val_tokenized_skill = X_val.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

test_tokenized_skill = X_test.map(
    lambda x: tokenize_align(x, "tags_skill", skills2id),
    batched = True,
    remove_columns= columns_to_remove
)

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/3174 [00:00<?, ? examples/s]

Map:   0%|          | 0/3569 [00:00<?, ? examples/s]

In [21]:
skill_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels = len(id2skills),
    id2label = id2skills,
    label2id = skills2id
).to(device)

data_collator = DataCollatorForTokenClassification(tokenizer)

[transformers] loading configuration file config.json from cache at /Users/phidinh/.cache/huggingface/hub/models--bert-base-cased/snapshots/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "B",
    "1": "O",
    "2": "I"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B": 0,
    "I": 2,
    "O": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version":

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [11]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from seqeval.scheme import IOB2

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis = 2)

    true_predictions = [
        [id2skills[p] for (p,l) in zip(prediction, label) if l != -100]
         for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [id2skills[l] for (p,l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "Precision": precision_score(true_labels, true_predictions, mode = "strict", scheme= IOB2),
        "Recall": recall_score(true_labels, true_predictions, mode="strict", scheme=IOB2),
        "F1_Score": f1_score(true_labels,true_predictions, mode="strict", scheme=IOB2),
    }

In [14]:
from transformers import TrainingArguments, Trainer

In [22]:
training_args = TrainingArguments(
    output_dir= "../models/bert_skill",
    eval_strategy = "epoch",
    learning_rate= 2e-05,
    per_device_eval_batch_size= 16,
    per_device_train_batch_size=16,
    num_train_epochs= 3,
    weight_decay= 0.01,
    logging_steps=10
)

trainer_skill = Trainer(
    model = skill_model,
    args = training_args,
    train_dataset=train_tokenized_skill,
    eval_dataset= val_tokenized_skill,
    data_collator= data_collator,
    processing_class = tokenizer,
    compute_metrics= compute_metrics
)

print("Start Fine-Tuning Bert Skill Model")
trainer_skill.train()

[transformers] PyTorch: setting up devices


Start Fine-Tuning Bert Skill Model


[transformers] ***** Running training *****
[transformers]   Num examples = 4,809
[transformers]   Num Epochs = 3
[transformers]   Num update steps per epoch = 301
[transformers]   Instantaneous batch size per device = 16
[transformers]   Total train batch size (w. parallel, distributed & accumulation) = 16
[transformers]   Gradient Accumulation steps = 1
[transformers]   Total optimization steps = 903
[transformers]   Number of trainable parameters = 107,721,987
/opt/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [18]:
print(unique_skills)
print(id2skills)

{'B', 'O', 'I'}
{0: 'B', 1: 'O', 2: 'I'}
